# Ders 4B — Advanced Ensemble: Voting, Stacking, Feature-Ensemble ve MultiView

Bu notebook Step 7 sonrasındaki en iyi gatekeeper kararlarını kullanır ve performansı artırabilecek ileri model yapılarını dener.

Gatekeeper kararları:

- `ERa_BLA_assay`: `Tuned RF + rdkit + balanced_oversampling`
- `ERa_LUC_VM7_assay`: `RF sampling gatekeeper + all_available + negative_5x_oversampling`

Bu dosyada dört ana yapı denenir:

1. `Voting`: en iyi gatekeeper model + önceki güçlü modeller.
2. `Stacking`: önceki güçlü modeller + gatekeeper; `passthrough=False` ve `passthrough=True` varyantları.
3. `Top-4 Feature Ensemble`: farklı feature setleriyle eğitilmiş modellerin eşit ağırlıklı ortalaması.
4. `MultiView`: her feature set bir view olarak kullanılır; eşit ağırlıklı ve lineer meta-model varyantları denenir.

Sampling sadece train set üzerinde uygulanır. Test set doğal dağılımında kalır.

## 1. Paket kontrolü

Bu scriptte gerçekten kullanılacak paketler kontrol edilir. Gereksiz paket import edilmez.

Bu dosyada `scikit-learn` modelleriyle voting, stacking, feature-level ensemble ve multiview ensemble kurulacaktır.

In [ ]:
import sys
# Mevcut Python ortamında pip komutunu çalıştırmak için kullanılır.
import subprocess
# Eksik paketleri notebook içinden kurmak için kullanılır.
import importlib.util
# Bir paketin kurulu olup olmadığını kontrol etmek için kullanılır.

def install_if_missing(import_name, pip_name=None):
    """Eksik paket varsa kurar; paket zaten varsa işlem yapmaz."""
    pip_name = pip_name or import_name
    # pip adı verilmezse import adı paket adı olarak kullanılır.
    if importlib.util.find_spec(import_name) is None:
        # Paket kurulu değilse pip kurulumu yapılır.
        print(f"[INSTALL] {pip_name}")
        # Hangi paketin kurulacağı ekrana yazdırılır.
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        # Paket sessiz modda kurulur.
    else:
        print(f"[OK] {import_name}")
        # Paket zaten kuruluysa tekrar kurulmaz.

required_packages = [
    ("pandas", "pandas"),
    # CSV okuma ve tablo işlemleri için kullanılır.
    ("numpy", "numpy"),
    # Sayısal matris, skor ortalaması ve index işlemleri için kullanılır.
    ("matplotlib", "matplotlib"),
    # ROC-AUC bar chart çizimleri için kullanılır.
    ("sklearn", "scikit-learn"),
    # RandomForest, ExtraTrees, boosting, voting, stacking ve metrikler için kullanılır.
]
# Bu scriptte gerçekten kullanılan paketler listelenir.

for import_name, pip_name in required_packages:
    # Paketler tek tek kontrol edilir.
    install_if_missing(import_name, pip_name)
    # Eksik paket varsa kurulur.

print("Paket kontrolü tamamlandı.")
# Paket kontrolünün bittiği yazdırılır.

## 2. Importlar ve genel ayarlar

Bu hücrede yalnızca bu scriptte kullanılacak paketler ve sabitler çağırılır.

In [ ]:
from pathlib import Path
# Dosya ve klasör yollarını güvenli şekilde yönetmek için kullanılır.
import warnings
# Gereksiz uyarıları kontrol etmek için kullanılır.
warnings.filterwarnings("ignore")
# Notebook çıktısını sade tutmak için uyarılar gizlenir.

import numpy as np
# Feature matrisleri, skor ortalamaları, sampling indexleri ve multiview işlemleri için kullanılır.
import pandas as pd
# Feature CSV dosyalarını okumak ve sonuç tablolarını kaydetmek için kullanılır.
import matplotlib.pyplot as plt
# Ensemble sonuçlarını bar chart olarak çizmek için kullanılır.

from sklearn.model_selection import train_test_split
# Train/test ve inner validation ayrımı için kullanılır.
from sklearn.metrics import accuracy_score, balanced_accuracy_score, average_precision_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
# Model performansını ölçmek için gerekli metrikler çağırılır.
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    GradientBoostingClassifier,
    VotingClassifier,
    StackingClassifier,
)
# RF, ExtraTrees, boosting, voting ve stacking modelleri çağırılır.
from sklearn.linear_model import LogisticRegression
# Stacking final estimator ve multiview lineer meta-model için kullanılır.
from sklearn.impute import SimpleImputer
# Eksik değerleri train setten öğrenilen median değerlerle doldurmak için kullanılır.

DATASETS = {
    # İki veri setinin kısa isimleri ve GitHub raw feature dosyaları burada tanımlanır.
    "ERa_BLA_assay": {
        "short_name": "ERα BLA",
        "model_prefix": "model_ERa_BLA",
        "feature_url": "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/molfest_outputs/01_feature_store/model_ERa_BLA_features.csv",
        "raw_url": "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/Train_2_1_A14_A17_ERa_BLA_agonist_antagonist.csv",
    },
    "ERa_LUC_VM7_assay": {
        "short_name": "ERα LUC VM7",
        "model_prefix": "model_ERa_LUC_VM7",
        "feature_url": "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/molfest_outputs/01_feature_store/model_ERa_LUC_VM7_features.csv",
        "raw_url": "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/Train_2_2_A15_A18_ERa_LUC_VM7_agonist_antagonist.csv",
    },
}

RANDOM_STATE = 42
# Train/test ayrımı, RandomForest ve sampling için sabit random seed belirlenir.
TEST_SIZE = 0.20
# Test set oranı %20 olarak belirlenir.
INNER_VALIDATION_SIZE = 0.25
# Feature-ensemble ve multiview lineer ağırlık için train içinde validation oranı belirlenir.
STACKING_CV_FOLDS = 3
# Stacking içinde 3-fold CV kullanılır.

RF_GATEKEEPER_TREES = 300
# Gatekeeper RandomForest için ağaç sayısı.
FAST_TREE_TREES = 120
# Voting/stacking içindeki yardımcı ağaç modelleri için daha hızlı ağaç sayısı.
FEATURE_ENSEMBLE_TREES = 80
# Feature ensemble sırasında çok sayıda model eğitileceği için daha düşük ağaç sayısı.
MULTIVIEW_TREES = 80
# Multiview sırasında çok sayıda view modeli eğitileceği için daha düşük ağaç sayısı.

OUTPUT_ROOT = Path("molfest_outputs")
# Bu scriptin çıktılarının kaydedileceği ana klasör belirlenir.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
# Çıktı klasörü yoksa oluşturulur.

print("Ayarlar hazır.")
# Ayar hücresinin tamamlandığı yazdırılır.
print("Feature CSV klasörü:", "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/molfest_outputs/01_feature_store")
# Hazır feature dosyalarının okunacağı GitHub raw klasörü gösterilir.

## 3. Yardımcı fonksiyonlar

Bu fonksiyonlar notebook içinde görünür durumdadır. Arka planda çalışan ayrı `utils.py` dosyası kullanılmaz.

In [ ]:
def show_table(df, n=50, title=None):
    """Sonuç tablosunu Colab'da tablo olarak, terminalde metin olarak gösterir."""
    if title:
        # Tablo başlığı varsa önce başlık yazdırılır.
        print("\n" + title)
    try:
        # Colab/Jupyter ortamında display fonksiyonu varsa tablo olarak gösterilir.
        display(df.head(n))
    except NameError:
        # Terminalde çalıştırılırsa tablo metin olarak yazdırılır.
        print(df.head(n).to_string(index=False))


def load_feature_table(dataset_key):
    """GitHub raw feature CSV dosyasını okur."""
    info = DATASETS[dataset_key]
    # Veri setine ait link ve kısa isim bilgileri alınır.
    df = pd.read_csv(info["feature_url"])
    # Hazır feature CSV dosyası GitHub raw linkinden okunur.
    print(f"\n{dataset_key} okundu: {df.shape[0]} satır, {df.shape[1]} kolon")
    # Okunan veri boyutu ekrana yazdırılır.
    return df
    # Okunan tablo döndürülür.


def feature_columns(df, feature_set):
    """İstenen feature ailesine göre kolon seçer."""
    prefix_map = {
        "morgan": ["Morgan_"],
        "maccs": ["MACCS_"],
        "rdkit": ["RDKit_"],
        "avalon": ["Avalon_"],
        "maccs_morgan": ["MACCS_", "Morgan_"],
        "maccs_rdkit": ["MACCS_", "RDKit_"],
        "morgan_rdkit": ["Morgan_", "RDKit_"],
        "avalon_rdkit": ["Avalon_", "RDKit_"],
        "maccs_avalon": ["MACCS_", "Avalon_"],
        "all_available": ["Morgan_", "MACCS_", "RDKit_", "Avalon_"],
    }
    # Feature set adları ile kolon prefixleri eşleştirilir.
    prefixes = prefix_map[feature_set]
    # İstenen feature set için prefix listesi alınır.
    cols = [c for c in df.columns if any(c.startswith(p) for p in prefixes)]
    # Seçilen prefixlerden biriyle başlayan kolonlar feature kolonu olarak alınır.
    if not cols:
        # Hiç feature kolonu bulunamazsa açık hata verilir.
        raise ValueError(f"{feature_set} için feature kolonu bulunamadı.")
    return cols
    # Seçilen feature kolon isimleri döndürülür.


def available_feature_sets(df):
    """Feature CSV içinde gerçekten bulunan feature setlerini döndürür."""
    candidate_sets = [
        "rdkit",
        "morgan",
        "maccs",
        "avalon",
        "maccs_rdkit",
        "morgan_rdkit",
        "avalon_rdkit",
        "maccs_morgan",
        "maccs_avalon",
        "all_available",
    ]
    # Denenebilecek tekil ve birleşik feature setleri tanımlanır.
    valid_sets = []
    # Gerçekten kolonu bulunan feature setleri burada tutulur.
    for fs in candidate_sets:
        # Her aday feature set sırayla kontrol edilir.
        try:
            cols = feature_columns(df, fs)
            # Feature kolonları bulunabiliyor mu kontrol edilir.
            if len(cols) > 0:
                # Kolon varsa feature set geçerli kabul edilir.
                valid_sets.append(fs)
                # Feature set listeye eklenir.
        except ValueError:
            # Kolon yoksa bu feature set atlanır.
            pass
    return valid_sets
    # Geçerli feature set listesi döndürülür.


def make_xy(df, cols):
    """Feature kolonlarından X matrisi ve Target kolonundan y vektörü üretir."""
    X = (
        df[cols]
        # Sadece seçilen feature kolonları alınır.
        .apply(pd.to_numeric, errors="coerce")
        # Sayısal olmayan değerler güvenli şekilde NaN yapılır.
        .replace([np.inf, -np.inf], np.nan)
        # Sonsuz değerler NaN'a çevrilir.
        .to_numpy(dtype=np.float32)
        # sklearn modelleri için numpy matrise çevrilir.
    )
    y = df["Target"].astype(int).to_numpy()
    # Binary target kolonu integer numpy vektörüne çevrilir.
    return X, y
    # Modelleme için X ve y döndürülür.


def split_indices(y):
    """Bütün modeller için aynı train/test indexlerini üretir."""
    idx = np.arange(len(y))
    # Satır indexleri oluşturulur.
    train_idx, test_idx = train_test_split(
        idx,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_STATE,
    )
    # Sınıf oranını koruyan train/test index ayrımı yapılır.
    return train_idx, test_idx
    # Aynı split bütün feature viewlarda kullanılmak üzere döndürülür.


def matrix_for_feature_set(df, feature_set, train_idx, test_idx):
    """Bir feature set için aynı train/test indexleriyle X_train, X_test üretir."""
    cols = feature_columns(df, feature_set)
    # İstenen feature setin kolonları alınır.
    X_all, y_all = make_xy(df, cols)
    # Tüm veri için X ve y hazırlanır.
    X_train = X_all[train_idx]
    # Train indexlerine karşılık gelen feature matrisi alınır.
    X_test = X_all[test_idx]
    # Test indexlerine karşılık gelen feature matrisi alınır.
    y_train = y_all[train_idx]
    # Train target vektörü alınır.
    y_test = y_all[test_idx]
    # Test target vektörü alınır.
    X_train, X_test = impute_train_test(X_train, X_test)
    # Eksik değerler yalnızca train setten öğrenilen median ile doldurulur.
    return X_train, X_test, y_train, y_test, cols
    # Modelleme matrisleri ve feature isimleri döndürülür.


def impute_train_test(X_train, X_test):
    """Eksik değerleri yalnızca train setten öğrenilen median değerlerle doldurur."""
    imputer = SimpleImputer(strategy="median")
    # SimpleImputer her feature için median değeri train setten öğrenir.
    X_train_imputed = imputer.fit_transform(X_train)
    # Imputer sadece train set üzerinde fit edilir; test bilgisi train'e sızmaz.
    X_test_imputed = imputer.transform(X_test)
    # Test set aynı imputer ile dönüştürülür.
    return X_train_imputed, X_test_imputed
    # Model eğitiminde kullanılacak temiz train/test matrisleri döndürülür.


def class1_score(model, X):
    """Modelden class 1 için skor veya olasılık üretir."""
    if hasattr(model, "predict_proba"):
        # Model olasılık üretebiliyorsa class 1 olasılığı alınır.
        score = model.predict_proba(X)[:, 1]
        # Class 1 olasılıkları alınır.
    elif hasattr(model, "decision_function"):
        # Olasılık yoksa karar fonksiyonu skoru kullanılır.
        score = model.decision_function(X)
        # Karar fonksiyonu skoru alınır.
    else:
        score = model.predict(X).astype(float)
        # Son seçenek olarak sınıf tahmini skor gibi kullanılır.

    score = np.asarray(score, dtype=float)
    # Skorlar numpy array formatına çevrilir.
    score = np.nan_to_num(score, nan=0.5, posinf=1.0, neginf=0.0)
    # ROC hesabı bozulmasın diye NaN/inf skorlar güvenli değerlere çekilir.
    return score
    # Temizlenmiş class 1 skoru döndürülür.


def metric_dict(y_true, y_pred, y_score):
    """Test tahminlerinden temel sınıflandırma metriklerini hesaplar."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    # Confusion matrix değerleri ayrı ayrı alınır.
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    # Specificity = TN / (TN + FP) olarak hesaplanır.
    return {
        "ROC": roc_auc_score(y_true, y_score),
        # ROC-AUC, pozitif ve negatif sınıf ayrım gücünü ölçer.
        "AP": average_precision_score(y_true, y_score),
        # AP, precision-recall eğrisi altındaki ortalama performanstır.
        "F1": f1_score(y_true, y_pred, zero_division=0),
        # F1, precision ve recall dengesini özetler.
        "Accuracy": accuracy_score(y_true, y_pred),
        # Accuracy, toplam doğru sınıflandırma oranıdır.
        "BalancedAccuracy": balanced_accuracy_score(y_true, y_pred),
        # Balanced accuracy, sınıf dengesizliğinde daha adil bir doğruluk ölçüsüdür.
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        # Recall = TP / (TP + FN) olarak hesaplanır.
        "Specificity": specificity,
        # Specificity = TN / (TN + FP) olarak hesaplanır.
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        # Precision = TP / (TP + FP) olarak hesaplanır.
        "TN": int(tn),
        # True Negative sayısı.
        "FP": int(fp),
        # False Positive sayısı.
        "FN": int(fn),
        # False Negative sayısı.
        "TP": int(tp),
        # True Positive sayısı.
    }


def make_rf(params=None, n_estimators=None, seed_offset=0):
    """RandomForest modeli kurar; parametre verilirse onları kullanır."""
    base_params = {
        "n_estimators": RF_GATEKEEPER_TREES if n_estimators is None else n_estimators,
        # Ağaç sayısı gatekeeper veya hızlı ensemble ayarına göre belirlenir.
        "max_features": "sqrt",
        # Her splitte featureların karekökü kadar aday denenir.
        "class_weight": "balanced_subsample",
        # Sınıf dengesizliğini her bootstrap örneğinde dengelemeye çalışır.
        "random_state": RANDOM_STATE + seed_offset,
        # Tekrarlanabilirlik sağlar.
        "n_jobs": -1,
        # Mevcut işlemciler kullanılır.
    }
    # Varsayılan RF parametreleri tanımlanır.
    if params is not None:
        # Tuning sonucundan gelen parametreler varsa kullanılır.
        base_params.update(params)
        # Tuning parametreleri varsayılanların üzerine yazılır.
        if n_estimators is not None:
            # Hızlı varyant istenirse ağaç sayısı tekrar sınırlandırılır.
            base_params["n_estimators"] = n_estimators
        base_params["random_state"] = RANDOM_STATE + seed_offset
        # Random state sabit tutulur.
        base_params["n_jobs"] = -1
        # Paralel işlem korunur.
    return RandomForestClassifier(**base_params)
    # RandomForest modeli döndürülür.


def make_strong_model_set(gatekeeper_params, include_gatekeeper=True):
    """Voting/stacking için güçlü klasik model setini oluşturur."""
    models = []
    # Model listesi burada hazırlanır.
    if include_gatekeeper:
        # İstenirse en güçlü mevcut gatekeeper model de ensemble içine eklenir.
        models.append(("gatekeeper_rf", make_rf(gatekeeper_params, seed_offset=0)))
        # Gatekeeper RF modeli listeye eklenir.

    models.append((
        "extra_trees",
        ExtraTreesClassifier(
            n_estimators=FAST_TREE_TREES,
            # Yardımcı model için hızlı ama stabil ağaç sayısı kullanılır.
            max_features="sqrt",
            # Her splitte featureların karekökü kadar aday denenir.
            class_weight="balanced",
            # Sınıf dengesizliği için dengeli ağırlık kullanılır.
            random_state=RANDOM_STATE + 11,
            # Tekrarlanabilirlik sağlar.
            n_jobs=-1,
            # Paralel işlem kullanılır.
        )
    ))
    # Önceki karşılaştırmalarda güçlü çıkan ExtraTrees modeli eklenir.

    models.append((
        "hist_gradient_boosting",
        HistGradientBoostingClassifier(
            max_iter=120,
            # Boosting iterasyonu sınırlı tutularak süre kontrol edilir.
            learning_rate=0.06,
            # Kontrollü öğrenme oranı kullanılır.
            l2_regularization=0.01,
            # Hafif regularizasyon uygulanır.
            random_state=RANDOM_STATE + 22,
            # Tekrarlanabilirlik sağlar.
        )
    ))
    # Önceki karşılaştırmalarda güçlü çıkan HistGradientBoosting modeli eklenir.

    models.append((
        "gradient_boosting",
        GradientBoostingClassifier(
            n_estimators=120,
            # Klasik gradient boosting için hızlı ağaç sayısı seçilir.
            learning_rate=0.05,
            # Kontrollü öğrenme oranı kullanılır.
            max_depth=3,
            # Sığ ağaçlar kullanılır.
            random_state=RANDOM_STATE + 33,
            # Tekrarlanabilirlik sağlar.
        )
    ))
    # Gatekeeper dışı bir boosting alternatifi daha eklenir.

    return models
    # Voting/stacking için model listesi döndürülür.


def resample_indices(y_train, ratio, method):
    """Train set üzerinde over/under sampling için seçilecek indeksleri üretir."""
    if method == "none":
        # Sampling yoksa train set olduğu gibi kullanılır.
        return np.arange(len(y_train))

    rng = np.random.RandomState(RANDOM_STATE)
    # Sampling işleminin tekrarlanabilir olması için sabit random generator oluşturulur.
    pos = np.where(y_train == 1)[0]
    # Pozitif sınıf indexleri alınır.
    neg = np.where(y_train == 0)[0]
    # Negatif sınıf indexleri alınır.
    current_ratio = len(pos) / len(neg)
    # Mevcut pozitif/negatif oranı hesaplanır.

    if method == "oversampling":
        # Oversampling az olan sınıfı çoğaltır.
        if current_ratio < ratio:
            n_pos, n_neg = int(np.ceil(ratio * len(neg))), len(neg)
            # Hedef oran için pozitif sınıf çoğaltılır.
        else:
            n_pos, n_neg = len(pos), int(np.ceil(len(pos) / ratio))
            # Hedef oran için negatif sınıf çoğaltılır.
        selected_pos = rng.choice(pos, n_pos, replace=n_pos > len(pos))
        # Pozitif örnekler gerekirse tekrarlı seçilir.
        selected_neg = rng.choice(neg, n_neg, replace=n_neg > len(neg))
        # Negatif örnekler gerekirse tekrarlı seçilir.

    elif method == "undersampling":
        # Undersampling fazla olan sınıfı azaltır.
        if current_ratio < ratio:
            n_pos = len(pos)
            # Pozitif sınıf korunur.
            n_neg = min(len(neg), max(5, int(np.floor(len(pos) / ratio))))
            # Negatif sınıf hedef orana göre azaltılır.
        else:
            n_pos = min(len(pos), max(5, int(np.floor(ratio * len(neg)))))
            # Pozitif sınıf hedef orana göre azaltılır.
            n_neg = len(neg)
            # Negatif sınıf korunur.
        selected_pos = rng.choice(pos, n_pos, replace=False)
        # Pozitif sınıf tekrarsız örneklenir.
        selected_neg = rng.choice(neg, n_neg, replace=False)
        # Negatif sınıf tekrarsız örneklenir.

    else:
        raise ValueError("method: none, oversampling veya undersampling olmalı.")
        # Bilinmeyen sampling yöntemi için açık hata verilir.

    idx = np.concatenate([selected_pos, selected_neg])
    # Pozitif ve negatif indexler birleştirilir.
    rng.shuffle(idx)
    # Train sırası karıştırılır.
    return idx
    # Sampling uygulanmış train indexleri döndürülür.


def evaluate_from_score(y_true, y_score):
    """Skor vektöründen sınıf tahmini ve metrik üretir."""
    y_score = np.nan_to_num(np.asarray(y_score, dtype=float), nan=0.5, posinf=1.0, neginf=0.0)
    # Skorlar güvenli sayısal değerlere çevrilir.
    y_pred = (y_score >= 0.5).astype(int)
    # 0.5 eşik değeriyle sınıf tahmini üretilir.
    return y_pred, metric_dict(y_true, y_pred, y_score)
    # Tahmin ve metrikler döndürülür.


def plot_metric_bar(df, label_col, metric, title, out_file=None):
    """Performans sonuçlarını yatay bar chart olarak çizer."""
    plot_df = df.sort_values(metric, ascending=False).copy()
    # Sonuçlar metrik değerine göre sıralanır.
    plot_df = plot_df.sort_values(metric, ascending=True)
    # Yatay bar chartta en iyi değer üstte görünsün diye çizim öncesi ters sıralanır.

    plt.figure(figsize=(9, max(4, 0.40 * len(plot_df))))
    # Grafik boyutu satır sayısına göre ayarlanır.
    plt.barh(plot_df[label_col].astype(str), plot_df[metric])
    # Her aday için metrik değeri yatay bar olarak çizilir.
    plt.xlabel(metric)
    # X eksenine hangi metrik çizildiği yazılır.
    if metric == "ROC":
        # ROC-AUC farkları daha net görünsün diye ROC grafikleri 0.60'tan başlatılır.
        plt.xlim(left=0.60)
    plt.title(title)
    # Grafiğe açıklayıcı başlık eklenir.
    plt.tight_layout()
    # Etiketlerin kesilmemesi için yerleşim düzenlenir.
    if out_file:
        # Dosya yolu verilmişse grafik kaydedilir.
        Path(out_file).parent.mkdir(parents=True, exist_ok=True)
        # Kayıt klasörü yoksa oluşturulur.
        plt.savefig(out_file, dpi=300, bbox_inches="tight")
        # Grafik yüksek çözünürlüklü PNG olarak kaydedilir.
    plt.show()
    # Grafik ekranda gösterilir.

## 4. Gatekeeper ayarları

Step 7 sonucuna göre devam edilecek gatekeeper ayarları sabitlenir.

BLA tarafında tuned RF test ROC-AUC açısından geçtiği için tuned parametreler kullanılır.  
LUC VM7 tarafında tuned RF ROC-AUC açısından geçemediği için sampling gatekeeper korunur.

In [ ]:
gatekeeper_configs = {
    "ERa_BLA_assay": {
        "Comparison": "Tuned RF gatekeeper",
        "FeatureSet": "rdkit",
        "SamplingScenario": "balanced_oversampling",
        "SamplingMethod": "oversampling",
        "SamplingRatio": 1.0,
        "RFParams": {
            "n_estimators": 300,
            "min_samples_split": 5,
            "min_samples_leaf": 2,
            "max_features": "sqrt",
            "max_depth": 15,
            "criterion": "entropy",
            "class_weight": "balanced_subsample",
            "bootstrap": True,
        },
    },
    "ERa_LUC_VM7_assay": {
        "Comparison": "RF sampling gatekeeper",
        "FeatureSet": "all_available",
        "SamplingScenario": "negative_5x_oversampling",
        "SamplingMethod": "oversampling",
        "SamplingRatio": 0.2,
        "RFParams": None,
    },
}
# Step 7 sonrasındaki gatekeeper kararları sabitlenir.

show_table(
    pd.DataFrame([
        {
            "Dataset": dataset_key,
            "Gatekeeper": cfg["Comparison"],
            "FeatureSet": cfg["FeatureSet"],
            "SamplingScenario": cfg["SamplingScenario"],
            "SamplingMethod": cfg["SamplingMethod"],
            "SamplingRatio": cfg["SamplingRatio"],
            "UsesTunedParams": cfg["RFParams"] is not None,
        }
        for dataset_key, cfg in gatekeeper_configs.items()
    ]),
    title="Step 8 için kullanılan gatekeeper ayarları"
)
# Gatekeeper ayarları ekranda gösterilir.

## 5. Voting, stacking, top-4 feature ensemble ve multiview modeller

Bu hücre bütün ileri model yapılarını kurar ve gatekeeper ile karşılaştırır.

Denenen yapıların özeti:

- `Voting_With_Gatekeeper`: gatekeeper RF + ExtraTrees + HistGradientBoosting + GradientBoosting.
- `Voting_Without_Gatekeeper`: gatekeeper olmadan güçlü alternatif modeller.
- `Stacking_LR_Meta_Passthrough_False`: base modeller + logistic regression meta-model.
- `Stacking_RF_Meta_Passthrough_True`: base modeller + orijinal featurelar + RF meta-model.
- `Top4_FeatureSet_EqualWeight`: iç validation ile seçilen 4 feature set modelinin eşit ağırlıklı ortalaması.
- `MultiView_EqualWeight`: her feature set bir view; skorlar eşit ağırlıkla ortalanır.
- `MultiView_LinearMeta`: view skorları train içi validation üzerinde logistic regression ile lineer birleştirilir.

In [ ]:
lesson_out = OUTPUT_ROOT / "08_advanced_ensembles"
# Advanced ensemble sonuçları için çıktı klasörü belirlenir.
lesson_out.mkdir(parents=True, exist_ok=True)
# Çıktı klasörü yoksa oluşturulur.

all_results = []
# Bütün model sonuçları burada tutulur.
all_feature_view_rows = []
# Top-4 feature set ve multiview iç validation sonuçları burada tutulur.
comparison_rows = []
# Gatekeeper vs en iyi advanced model karşılaştırmaları burada tutulur.

for dataset_key in DATASETS:
    # Her veri seti sırayla işlenir.
    df = load_feature_table(dataset_key)
    # Hazır feature CSV dosyası okunur.

    cfg = gatekeeper_configs[dataset_key]
    # Bu veri setinin gatekeeper ayarları alınır.
    y_all = df["Target"].astype(int).to_numpy()
    # Aynı train/test split için target vektörü alınır.
    train_idx, test_idx = split_indices(y_all)
    # Bütün modeller ve bütün feature setleri için ortak train/test indexleri üretilir.

    X_train, X_test, y_train, y_test, selected_features = matrix_for_feature_set(
        df,
        cfg["FeatureSet"],
        train_idx,
        test_idx,
    )
    # Gatekeeper feature setiyle train/test matrisleri hazırlanır.

    sampled_idx = resample_indices(y_train, ratio=cfg["SamplingRatio"], method=cfg["SamplingMethod"])
    # Sampling indexleri sadece train set üzerinden üretilir.
    X_train_sampled = X_train[sampled_idx]
    # Sampling uygulanmış train feature matrisi hazırlanır.
    y_train_sampled = y_train[sampled_idx]
    # Sampling uygulanmış train target vektörü hazırlanır.

    sampled_counts = dict(pd.Series(y_train_sampled).value_counts().sort_index())
    # Sampling sonrası train sınıf dağılımı hesaplanır.
    print(f"{dataset_key} | gatekeeper: {cfg['Comparison']} | sampled train distribution: {sampled_counts}")
    # Sampling sonrası train dağılımı ekrana yazdırılır.

    gatekeeper_model = make_rf(cfg["RFParams"])
    # Seçilen gatekeeper RF modeli kurulur.
    gatekeeper_model.fit(X_train_sampled, y_train_sampled)
    # Gatekeeper modeli sampling uygulanmış train set üzerinde eğitilir.
    y_pred = gatekeeper_model.predict(X_test)
    # Doğal test seti için sınıf tahmini alınır.
    y_score = class1_score(gatekeeper_model, X_test)
    # ROC/AP için class 1 skoru alınır.
    gatekeeper_metrics = metric_dict(y_test, y_pred, y_score)
    # Test performans metrikleri hesaplanır.
    gatekeeper_metrics.update({
        "Dataset": dataset_key,
        "ModelFamily": "Gatekeeper",
        "ModelName": cfg["Comparison"],
        "FeatureStrategy": cfg["FeatureSet"],
        "FeatureSet": cfg["FeatureSet"],
        "SamplingScenario": cfg["SamplingScenario"],
        "SamplingMethod": cfg["SamplingMethod"],
        "SamplingRatio": cfg["SamplingRatio"],
        "FeatureCount": len(selected_features),
        "TrainClass0_after_sampling": int(sampled_counts.get(0, 0)),
        "TrainClass1_after_sampling": int(sampled_counts.get(1, 0)),
    })
    # Gatekeeper sonucuna model, feature ve sampling bilgileri eklenir.
    all_results.append(gatekeeper_metrics)
    # Gatekeeper sonucu genel listeye eklenir.

    # ============================================================
    # 1) Voting modelleri
    # ============================================================
    voting_with_gatekeeper = VotingClassifier(
        estimators=make_strong_model_set(cfg["RFParams"], include_gatekeeper=True),
        # Gatekeeper RF + önceki güçlü modeller birlikte kullanılır.
        voting="soft",
        # Olasılık ortalaması yapılır.
        weights=[2, 1, 1, 1],
        # Gatekeeper modele daha yüksek ağırlık verilir.
        n_jobs=-1,
        # Paralel işlem kullanılır.
    )
    # Gatekeeper içeren soft voting modeli oluşturulur.

    voting_without_gatekeeper = VotingClassifier(
        estimators=make_strong_model_set(cfg["RFParams"], include_gatekeeper=False),
        # Gatekeeper olmadan önceki güçlü modeller kullanılır.
        voting="soft",
        # Olasılık ortalaması yapılır.
        n_jobs=-1,
        # Paralel işlem kullanılır.
    )
    # Gatekeeper içermeyen soft voting modeli oluşturulur.

    # ============================================================
    # 2) Stacking modelleri
    # ============================================================
    stacking_lr_false = StackingClassifier(
        estimators=make_strong_model_set(cfg["RFParams"], include_gatekeeper=True),
        # Gatekeeper RF + önceki güçlü modeller base estimator olarak kullanılır.
        final_estimator=LogisticRegression(max_iter=2000, solver="liblinear"),
        # Meta seviyede LogisticRegression kullanılır.
        stack_method="predict_proba",
        # Base modellerden olasılık çıktıları alınır.
        passthrough=False,
        # Meta modele orijinal featurelar verilmez.
        cv=STACKING_CV_FOLDS,
        # Stacking içinde 3-fold CV kullanılır.
        n_jobs=-1,
        # Paralel işlem kullanılır.
    )
    # Passthrough kapalı klasik stacking modeli oluşturulur.

    stacking_rf_true = StackingClassifier(
        estimators=make_strong_model_set(cfg["RFParams"], include_gatekeeper=False),
        # Base seviyede gatekeeper dışındaki güçlü modeller kullanılır.
        final_estimator=make_rf(cfg["RFParams"], n_estimators=FAST_TREE_TREES, seed_offset=99),
        # Passthrough=True olduğunda meta seviyede en güçlü RF ailesi kullanılır.
        stack_method="predict_proba",
        # Base modellerden olasılık çıktıları alınır.
        passthrough=True,
        # Meta modele hem base model olasılıkları hem de orijinal featurelar verilir.
        cv=STACKING_CV_FOLDS,
        # Stacking içinde 3-fold CV kullanılır.
        n_jobs=-1,
        # Paralel işlem kullanılır.
    )
    # Passthrough açık ve RF meta-model kullanan stacking modeli oluşturulur.

    same_feature_models = {
        "Voting_With_Gatekeeper": voting_with_gatekeeper,
        "Voting_Without_Gatekeeper": voting_without_gatekeeper,
        "Stacking_LR_Meta_Passthrough_False": stacking_lr_false,
        "Stacking_RF_Meta_Passthrough_True": stacking_rf_true,
    }
    # Aynı gatekeeper feature seti üzerinde denenecek advanced modeller listelenir.

    for model_name, model in same_feature_models.items():
        # Voting ve stacking modelleri sırayla denenir.
        model.fit(X_train_sampled, y_train_sampled)
        # Model sampling uygulanmış train set üzerinde eğitilir.
        y_pred = model.predict(X_test)
        # Doğal test seti için sınıf tahmini alınır.
        y_score = class1_score(model, X_test)
        # ROC/AP için class 1 skoru alınır.
        metrics = metric_dict(y_test, y_pred, y_score)
        # Test performans metrikleri hesaplanır.
        metrics.update({
            "Dataset": dataset_key,
            "ModelFamily": "SameFeatureAdvanced",
            "ModelName": model_name,
            "FeatureStrategy": cfg["FeatureSet"],
            "FeatureSet": cfg["FeatureSet"],
            "SamplingScenario": cfg["SamplingScenario"],
            "SamplingMethod": cfg["SamplingMethod"],
            "SamplingRatio": cfg["SamplingRatio"],
            "FeatureCount": len(selected_features),
            "TrainClass0_after_sampling": int(sampled_counts.get(0, 0)),
            "TrainClass1_after_sampling": int(sampled_counts.get(1, 0)),
        })
        # Sonuç satırına model ve feature bilgileri eklenir.
        all_results.append(metrics)
        # Sonuç genel listeye eklenir.

    # ============================================================
    # 3) Top-4 feature set equal-weight ensemble
    # ============================================================
    candidate_feature_sets = available_feature_sets(df)
    # CSV içinde bulunan feature setleri alınır.
    inner_train_rel, inner_val_rel = train_test_split(
        np.arange(len(y_train)),
        test_size=INNER_VALIDATION_SIZE,
        stratify=y_train,
        random_state=RANDOM_STATE,
    )
    # Train set içinde feature set seçimi için inner validation ayrımı yapılır.

    feature_view_scores = []
    # Her feature set için inner validation ROC ve test skorları burada tutulur.

    for feature_set in candidate_feature_sets:
        # Her feature set ayrı model/view olarak denenir.
        X_train_fs, X_test_fs, y_train_fs, y_test_fs, fs_cols = matrix_for_feature_set(
            df,
            feature_set,
            train_idx,
            test_idx,
        )
        # Bu feature set için aynı outer split ile matrisler hazırlanır.

        inner_sampled_rel = resample_indices(
            y_train_fs[inner_train_rel],
            ratio=cfg["SamplingRatio"],
            method=cfg["SamplingMethod"],
        )
        # Inner train parçasında sampling uygulanır.
        X_inner_sampled = X_train_fs[inner_train_rel][inner_sampled_rel]
        # Inner sampled feature matrisi hazırlanır.
        y_inner_sampled = y_train_fs[inner_train_rel][inner_sampled_rel]
        # Inner sampled target vektörü hazırlanır.

        inner_model = make_rf(cfg["RFParams"], n_estimators=FEATURE_ENSEMBLE_TREES, seed_offset=101)
        # Feature set seçimi için hızlı RF modeli kurulur.
        inner_model.fit(X_inner_sampled, y_inner_sampled)
        # Model inner sampled train üzerinde eğitilir.
        inner_val_score = class1_score(inner_model, X_train_fs[inner_val_rel])
        # Inner validation skorları alınır.
        inner_val_pred, inner_val_metrics = evaluate_from_score(y_train_fs[inner_val_rel], inner_val_score)
        # Inner validation ROC-AUC hesaplanır.

        full_sampled_rel = resample_indices(y_train_fs, ratio=cfg["SamplingRatio"], method=cfg["SamplingMethod"])
        # Outer train set üzerinde sampling uygulanır.
        full_model = make_rf(cfg["RFParams"], n_estimators=FEATURE_ENSEMBLE_TREES, seed_offset=202)
        # Test ensemble için full train üzerinde eğitilecek hızlı RF modeli kurulur.
        full_model.fit(X_train_fs[full_sampled_rel], y_train_fs[full_sampled_rel])
        # Model sampling uygulanmış outer train set üzerinde eğitilir.
        test_score = class1_score(full_model, X_test_fs)
        # Doğal test seti için class 1 skoru alınır.

        row = {
            "Dataset": dataset_key,
            "FeatureSet": feature_set,
            "FeatureCount": len(fs_cols),
            "InnerValidationROC": inner_val_metrics["ROC"],
            "InnerValidationAP": inner_val_metrics["AP"],
            "InnerValidationF1": inner_val_metrics["F1"],
            "SamplingScenario": cfg["SamplingScenario"],
            "ModelType": "RandomForest",
        }
        # Feature set seçimi için validation sonucu hazırlanır.
        all_feature_view_rows.append(row)
        # Feature view sonucu genel listeye eklenir.

        feature_view_scores.append({
            "FeatureSet": feature_set,
            "FeatureCount": len(fs_cols),
            "InnerValidationROC": inner_val_metrics["ROC"],
            "TestScore": test_score,
        })
        # Top-4 ensemble için test skorları ve validation performansı saklanır.

    feature_view_df = pd.DataFrame(feature_view_scores).sort_values("InnerValidationROC", ascending=False)
    # Feature setler inner validation ROC değerine göre sıralanır.
    top4_features = feature_view_df.head(4).copy()
    # En iyi 4 feature set seçilir.
    top4_scores = np.vstack(top4_features["TestScore"].to_list())
    # Top-4 feature set modellerinin test skorları matrise çevrilir.
    top4_equal_score = top4_scores.mean(axis=0)
    # Dört model 0.25 eşit ağırlıkla ortalanır.
    top4_pred, top4_metrics = evaluate_from_score(y_test, top4_equal_score)
    # Top-4 ensemble test metrikleri hesaplanır.
    top4_metrics.update({
        "Dataset": dataset_key,
        "ModelFamily": "FeatureSetEnsemble",
        "ModelName": "Top4_FeatureSet_EqualWeight",
        "FeatureStrategy": " + ".join(top4_features["FeatureSet"].tolist()),
        "FeatureSet": "Top4ValidationSelected",
        "SamplingScenario": cfg["SamplingScenario"],
        "SamplingMethod": cfg["SamplingMethod"],
        "SamplingRatio": cfg["SamplingRatio"],
        "FeatureCount": int(top4_features["FeatureCount"].sum()),
        "Top4FeatureSets": " | ".join(top4_features["FeatureSet"].tolist()),
        "Weights": "0.25 each",
    })
    # Top-4 ensemble sonucuna feature ve ağırlık bilgileri eklenir.
    all_results.append(top4_metrics)
    # Top-4 ensemble sonucu genel listeye eklenir.

    show_table(
        top4_features[["FeatureSet", "FeatureCount", "InnerValidationROC"]],
        title=f"{dataset_key} — Top-4 feature ensemble için seçilen feature setleri"
    )
    # Top-4 seçilen feature setleri ekranda gösterilir.

    # ============================================================
    # 4) MultiView equal-weight ve linear meta ensemble
    # ============================================================
    preferred_views = [fs for fs in ["morgan", "maccs", "rdkit", "avalon", "maccs_rdkit", "morgan_rdkit", "all_available"] if fs in candidate_feature_sets]
    # Multiview için tekil ve güçlü birleşik viewlar seçilir.

    view_test_scores = []
    # Her view modelinin test skorları burada tutulur.
    view_val_scores = []
    # Her view modelinin inner validation skorları burada tutulur.

    for view_name in preferred_views:
        # Her feature view ayrı model olarak eğitilir.
        X_train_view, X_test_view, y_train_view, y_test_view, view_cols = matrix_for_feature_set(
            df,
            view_name,
            train_idx,
            test_idx,
        )
        # Bu view için aynı outer split ile matrisler hazırlanır.

        inner_sampled_rel = resample_indices(
            y_train_view[inner_train_rel],
            ratio=cfg["SamplingRatio"],
            method=cfg["SamplingMethod"],
        )
        # Inner train parçasında sampling uygulanır.
        view_inner_model = make_rf(cfg["RFParams"], n_estimators=MULTIVIEW_TREES, seed_offset=303)
        # Inner validation view modeli kurulur.
        view_inner_model.fit(X_train_view[inner_train_rel][inner_sampled_rel], y_train_view[inner_train_rel][inner_sampled_rel])
        # Inner sampled train üzerinde view modeli eğitilir.
        val_score = class1_score(view_inner_model, X_train_view[inner_val_rel])
        # Inner validation set için view skoru alınır.
        view_val_scores.append(val_score)
        # Lineer meta-model için validation skoru saklanır.

        full_sampled_rel = resample_indices(y_train_view, ratio=cfg["SamplingRatio"], method=cfg["SamplingMethod"])
        # Outer train set üzerinde sampling uygulanır.
        view_full_model = make_rf(cfg["RFParams"], n_estimators=MULTIVIEW_TREES, seed_offset=404)
        # Test skoru için full train üzerinde eğitilecek view modeli kurulur.
        view_full_model.fit(X_train_view[full_sampled_rel], y_train_view[full_sampled_rel])
        # View modeli sampling uygulanmış outer train set üzerinde eğitilir.
        test_score = class1_score(view_full_model, X_test_view)
        # Doğal test seti için view skoru alınır.
        view_test_scores.append(test_score)
        # Equal-weight ve linear meta test matrisi için view skoru saklanır.

    view_test_matrix = np.vstack(view_test_scores).T
    # Test setindeki her molekül için view skorları matrise çevrilir.
    view_val_matrix = np.vstack(view_val_scores).T
    # Inner validation setindeki her molekül için view skorları matrise çevrilir.

    multiview_equal_score = view_test_matrix.mean(axis=1)
    # Multiview equal-weight modelinde view skorları eşit ağırlıkla ortalanır.
    multiview_equal_pred, multiview_equal_metrics = evaluate_from_score(y_test, multiview_equal_score)
    # Multiview equal-weight test metrikleri hesaplanır.
    multiview_equal_metrics.update({
        "Dataset": dataset_key,
        "ModelFamily": "MultiView",
        "ModelName": "MultiView_EqualWeight",
        "FeatureStrategy": " | ".join(preferred_views),
        "FeatureSet": "MultiView",
        "SamplingScenario": cfg["SamplingScenario"],
        "SamplingMethod": cfg["SamplingMethod"],
        "SamplingRatio": cfg["SamplingRatio"],
        "FeatureCount": len(preferred_views),
        "Views": " | ".join(preferred_views),
        "Weights": "equal",
    })
    # Multiview equal-weight sonucuna view bilgileri eklenir.
    all_results.append(multiview_equal_metrics)
    # Multiview equal-weight sonucu genel listeye eklenir.

    meta_model = LogisticRegression(max_iter=2000, solver="liblinear")
    # View skorlarını lineer şekilde birleştirmek için logistic regression meta-model kurulur.
    meta_model.fit(view_val_matrix, y_train[inner_val_rel])
    # Meta-model inner validation view skorları üzerinde eğitilir.
    multiview_linear_score = meta_model.predict_proba(view_test_matrix)[:, 1]
    # Doğal test seti için lineer meta-model olasılığı alınır.
    multiview_linear_pred, multiview_linear_metrics = evaluate_from_score(y_test, multiview_linear_score)
    # Multiview lineer meta-model test metrikleri hesaplanır.
    multiview_linear_metrics.update({
        "Dataset": dataset_key,
        "ModelFamily": "MultiView",
        "ModelName": "MultiView_LinearMeta",
        "FeatureStrategy": " | ".join(preferred_views),
        "FeatureSet": "MultiView",
        "SamplingScenario": cfg["SamplingScenario"],
        "SamplingMethod": cfg["SamplingMethod"],
        "SamplingRatio": cfg["SamplingRatio"],
        "FeatureCount": len(preferred_views),
        "Views": " | ".join(preferred_views),
        "Weights": str(meta_model.coef_.round(4).tolist()),
    })
    # Multiview lineer meta sonucuna view ve ağırlık bilgileri eklenir.
    all_results.append(multiview_linear_metrics)
    # Multiview lineer meta sonucu genel listeye eklenir.

    # ============================================================
    # Veri seti bazında sonuçların gösterimi
    # ============================================================
    dataset_result_df = pd.DataFrame([r for r in all_results if r["Dataset"] == dataset_key])
    # Bu veri setindeki bütün advanced model sonuçları tabloya çevrilir.
    dataset_result_df = dataset_result_df.sort_values("ROC", ascending=False).reset_index(drop=True)
    # Sonuçlar ROC-AUC değerine göre sıralanır.

    plot_metric_bar(
        dataset_result_df,
        label_col="ModelName",
        metric="ROC",
        title=f"{dataset_key} — advanced ensemble ROC-AUC karşılaştırması",
        out_file=lesson_out / f"{dataset_key}_advanced_ensemble_roc.png",
    )
    # Veri setindeki bütün advanced modeller ROC-AUC bar chart olarak çizilir.

    show_table(
        dataset_result_df[
            [
                "Dataset",
                "ModelFamily",
                "ModelName",
                "FeatureStrategy",
                "SamplingScenario",
                "ROC",
                "AP",
                "F1",
                "BalancedAccuracy",
            ]
        ],
        title=f"{dataset_key} — advanced ensemble sonuçları"
    )
    # Veri seti advanced sonuç tablosu ekranda gösterilir.

    best_advanced = dataset_result_df[dataset_result_df["ModelFamily"] != "Gatekeeper"].iloc[0].to_dict()
    # Gatekeeper dışındaki en iyi advanced model ROC-AUC'ye göre seçilir.

    reference_row = {
        "Dataset": dataset_key,
        "Comparison": "Gatekeeper",
        "ModelName": cfg["Comparison"],
        "ModelFamily": "Gatekeeper",
        "FeatureStrategy": cfg["FeatureSet"],
        "SamplingScenario": cfg["SamplingScenario"],
        "ROC": gatekeeper_metrics["ROC"],
        "AP": gatekeeper_metrics["AP"],
        "F1": gatekeeper_metrics["F1"],
        "ROC_Delta_vs_Gatekeeper": 0.0,
        "ROC_Gain_%": 0.0,
        "AP_Delta_vs_Gatekeeper": 0.0,
        "AP_Gain_%": 0.0,
        "F1_Delta_vs_Gatekeeper": 0.0,
        "F1_Gain_%": 0.0,
    }
    # Karşılaştırma tablosunun ilk satırı seçilen gatekeeper sonucudur.

    advanced_row = {
        "Dataset": dataset_key,
        "Comparison": "Best advanced ensemble",
        "ModelName": best_advanced["ModelName"],
        "ModelFamily": best_advanced["ModelFamily"],
        "FeatureStrategy": best_advanced["FeatureStrategy"],
        "SamplingScenario": best_advanced["SamplingScenario"],
        "ROC": best_advanced["ROC"],
        "AP": best_advanced["AP"],
        "F1": best_advanced["F1"],
        "ROC_Delta_vs_Gatekeeper": best_advanced["ROC"] - gatekeeper_metrics["ROC"],
        "ROC_Gain_%": 100 * (best_advanced["ROC"] - gatekeeper_metrics["ROC"]) / abs(gatekeeper_metrics["ROC"]) if abs(gatekeeper_metrics["ROC"]) > 1e-12 else np.nan,
        "AP_Delta_vs_Gatekeeper": best_advanced["AP"] - gatekeeper_metrics["AP"],
        "AP_Gain_%": 100 * (best_advanced["AP"] - gatekeeper_metrics["AP"]) / abs(gatekeeper_metrics["AP"]) if abs(gatekeeper_metrics["AP"]) > 1e-12 else np.nan,
        "F1_Delta_vs_Gatekeeper": best_advanced["F1"] - gatekeeper_metrics["F1"],
        "F1_Gain_%": 100 * (best_advanced["F1"] - gatekeeper_metrics["F1"]) / abs(gatekeeper_metrics["F1"]) if abs(gatekeeper_metrics["F1"]) > 1e-12 else np.nan,
    }
    # Karşılaştırma tablosunun ikinci satırı en iyi advanced ensemble sonucudur.

    comparison_rows.extend([reference_row, advanced_row])
    # İki satırlık karşılaştırma genel listeye eklenir.

all_results_df = pd.DataFrame(all_results).sort_values(["Dataset", "ROC"], ascending=[True, False])
# Bütün advanced model sonuçları tabloya çevrilir.
feature_view_df = pd.DataFrame(all_feature_view_rows).sort_values(["Dataset", "InnerValidationROC"], ascending=[True, False])
# Feature view validation sonuçları tabloya çevrilir.
comparison_df = pd.DataFrame(comparison_rows)
# Gatekeeper vs en iyi advanced karşılaştırmaları tabloya çevrilir.

all_results_df.to_csv(lesson_out / "08_all_advanced_ensemble_results.csv", index=False)
# Bütün advanced ensemble sonuçları CSV olarak kaydedilir.
feature_view_df.to_csv(lesson_out / "08_feature_view_inner_validation_results.csv", index=False)
# Feature view inner validation sonuçları CSV olarak kaydedilir.
comparison_df.to_csv(lesson_out / "08_gatekeeper_vs_best_advanced_ensemble.csv", index=False)
# Gatekeeper vs en iyi advanced ensemble karşılaştırmaları CSV olarak kaydedilir.

for dataset_key in DATASETS:
    # Her veri seti için ayrı final blok gösterilir.
    print("\n" + "#" * 100)
    # İki pipeline sonucunu görsel olarak ayırmak için keskin ayraç basılır.
    print(f"PIPELINE: {dataset_key}")
    # Hangi veri seti/pipeline raporlandığı yazdırılır.
    print("#" * 100)
    # Ayraç tamamlanır.

    dataset_comparison = comparison_df[comparison_df["Dataset"] == dataset_key].copy()
    # İlgili veri setinin gatekeeper vs en iyi advanced model karşılaştırması alınır.
    dataset_comparison.to_csv(lesson_out / f"{dataset_key}_gatekeeper_vs_best_advanced_ensemble.csv", index=False)
    # Veri setine özel final karşılaştırma tablosu CSV olarak kaydedilir.

    show_table(
        dataset_comparison[
            [
                "Dataset",
                "Comparison",
                "ModelName",
                "ModelFamily",
                "FeatureStrategy",
                "SamplingScenario",
                "ROC",
                "AP",
                "F1",
                "ROC_Delta_vs_Gatekeeper",
                "ROC_Gain_%",
                "AP_Delta_vs_Gatekeeper",
                "AP_Gain_%",
                "F1_Delta_vs_Gatekeeper",
                "F1_Gain_%",
            ]
        ],
        title=f"{dataset_key} — gatekeeper vs en iyi advanced ensemble"
    )
    # Asıl karar tablosu gösterilir: advanced yapı gatekeeper'ı geçti mi geçmedi mi burada görülür.

    dataset_results = all_results_df[all_results_df["Dataset"] == dataset_key].copy()
    # İlgili pipeline için bütün advanced model sonuçları alınır.
    dataset_results.to_csv(lesson_out / f"{dataset_key}_all_advanced_ensemble_results.csv", index=False)
    # Veri setine özel bütün advanced sonuçları CSV olarak kaydedilir.

    show_table(
        dataset_results[
            [
                "Dataset",
                "ModelFamily",
                "ModelName",
                "FeatureStrategy",
                "SamplingScenario",
                "ROC",
                "AP",
                "F1",
                "BalancedAccuracy",
            ]
        ],
        n=30,
        title=f"{dataset_key} — bütün advanced ensemble sonuçları"
    )
    # Bütün advanced ensemble sonuçları artık iki pipeline için ayrı ayrı gösterilir.

    dataset_feature_views = feature_view_df[feature_view_df["Dataset"] == dataset_key].copy()
    # İlgili pipeline için feature set / view iç validation sıralaması alınır.
    dataset_feature_views.to_csv(lesson_out / f"{dataset_key}_feature_view_inner_validation_results.csv", index=False)
    # Veri setine özel feature view sıralaması CSV olarak kaydedilir.

    show_table(
        dataset_feature_views[
            [
                "Dataset",
                "FeatureSet",
                "FeatureCount",
                "InnerValidationROC",
                "InnerValidationAP",
                "InnerValidationF1",
                "SamplingScenario",
            ]
        ],
        n=30,
        title=f"{dataset_key} — feature set / view iç validation sıralaması"
    )
    # Feature set ve view sıralamaları artık iki pipeline için ayrı ayrı gösterilir.
